# Beispiele: Strategien

Daten vorbereiten, Cross-Validation, Feature Selection, Hyperparameter-Suche, Metriken und Data Leakage. Datensatz für die meisten Beispiele: `load_diabetes`.

- **Herkunft:** selbst erstellt für die Lernapp, kein HSLU-Kursmaterial. Die Daten sind in scikit-learn eingebaut oder werden erzeugt.
- **Ausführen:** von oben nach unten, das Notebook hängt von keinem anderen ab. Lokal in der Lernapp oder auf jedem anderen PC im Browser: [In Colab öffnen](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/beispiele/06_Strategien.ipynb) (ein Google-Konto genügt, solange das Repo öffentlich ist; ist es privat, in Colab mit GitHub anmelden und «Include private repos» anhaken).
- **Navigation:** Das Inhaltsverzeichnis links in JupyterLab springt zu jedem Abschnitt. Zurück zur Übersicht: Lernapp, Kapitel «Übersicht».

**Mehr dazu in eigenen Notebooks:**
- [Fehlende_Werte_Imputation.ipynb](../Fehlende_Werte_Imputation.ipynb) ([in Colab](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/Fehlende_Werte_Imputation.ipynb)): Imputation ohne Data Leakage, mit 10 Wohnungen von Hand
- [Workflow_Train_Val_Test_CV.ipynb](../Workflow_Train_Val_Test_CV.ipynb) ([in Colab](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/Workflow_Train_Val_Test_CV.ipynb)): der ganze Ablauf: Split, Cross-Validation, Leakage-Vergleich, Klassifikation, Zeitreihen
- [Supervised_Learning_1_Uebungen_own.ipynb](../Supervised_Learning_1_Uebungen_own.ipynb) ([in Colab](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/Supervised_Learning_1_Uebungen_own.ipynb)): Übungen mit Lücken zu Imputation, One-Hot, Cross-Validation, GridSearch und Feature Selection

In [ ]:
# Colab has scikit-learn, pandas and matplotlib; umap and imblearn are installed there on demand
import importlib.util, subprocess, sys
fehlend = [pip for mod, pip in [("umap", "umap-learn"), ("imblearn", "imbalanced-learn")] if importlib.util.find_spec(mod) is None]
if fehlend and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *fehlend], check=True)
elif fehlend:
    print("Für einzelne Beispiele fehlt: pip install " + " ".join(fehlend))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import (load_diabetes, load_breast_cancer, load_wine, load_digits,
                              make_moons, make_blobs, make_classification)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score, silhouette_score,
                             adjusted_rand_score)

SEED = 42
plt.rcParams["figure.figsize"] = (6, 3.5)
pd.set_option("display.precision", 3)

## Daten

In [ ]:
X, y = load_diabetes(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

## Daten vorbereiten

### Fehlende Werte in der Pipeline

10 % der Werte von `bmi` werden gelöscht. Der `SimpleImputer` füllt sie mit dem Median, und zwar **in** der Pipeline, damit jede Cross-Validation-Runde nur ihre Trainingsdaten sieht.

In [ ]:
X_missing = X_train.copy()
rows = np.random.default_rng(SEED).choice(len(X_missing), size=len(X_missing) // 10, replace=False)
X_missing.iloc[rows, X_missing.columns.get_loc("bmi")] = np.nan

pipe_imp = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
scores_imp = cross_val_score(pipe_imp, X_missing, y_train, cv=5, scoring="r2")
print(f"R² CV mit Imputation: {scores_imp.mean():.3f} ± {scores_imp.std():.3f}")

### Kategorien: One-Hot-Encoding

Städte als 1, 2, 3 zu codieren würde eine Reihenfolge und Abstände erfinden (Bern = 2 × Luzern?). One-Hot macht pro Stadt eine eigene 0/1-Spalte.

In [ ]:
kunden = pd.DataFrame({"stadt": ["Luzern", "Zug", "Luzern", "Bern"], "umsatz": [10, 20, 30, 40]})
kunden_ohe = pd.get_dummies(kunden, columns=["stadt"])
kunden_ohe

### Alles zusammen: ColumnTransformer

Zahlen ergänzen und skalieren, Kategorien one-hot codieren, dann das Modell, alles in einer Pipeline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

rng = np.random.default_rng(SEED)
wohnungen = pd.DataFrame({
    "qm": rng.normal(80, 20, 200).round(),
    "zimmer": rng.integers(1, 6, 200).astype(float),
    "stadt": rng.choice(["Luzern", "Zug", "Bern"], 200),
})
wohnungen.loc[rng.choice(200, 15, replace=False), "qm"] = np.nan
preis = 15 * wohnungen.qm.fillna(80) + 100 * wohnungen.zimmer + wohnungen.stadt.map({"Luzern": 300, "Zug": 900, "Bern": 0}) \
        + rng.normal(0, 100, 200)

vorbereitung = ColumnTransformer([
    ("zahlen", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["qm", "zimmer"]),
    ("kategorien", OneHotEncoder(handle_unknown="ignore"), ["stadt"]),
])
pipe_ct = make_pipeline(vorbereitung, LinearRegression())
print(f"R² CV: {cross_val_score(pipe_ct, wohnungen, preis, cv=5, scoring='r2').mean():.3f}")

Ausführlich, Schritt für Schritt mit 10 Wohnungen: [Fehlende_Werte_Imputation.ipynb](../Fehlende_Werte_Imputation.ipynb).

## Train, Validation, Test und Cross-Validation

Kurzform: Pipeline aus Scaler und Modell, 5-fache Cross-Validation nur auf den Trainingsdaten.

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2")
print(scores.mean(), scores.std())

Der ganze Ablauf ausführlich (einfacher Split, Cross-Validation, Leakage-Vergleich, Klassifikation, Zeitreihen): [Workflow_Train_Val_Test_CV.ipynb](../Workflow_Train_Val_Test_CV.ipynb).

## Feature Selection

### Filter: Korrelation mit dem Ziel

In [ ]:
korr = X_train.corrwith(y_train).abs()
gewaehlt = korr[korr > 0.3].index.tolist()

pipe_fs = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
alle = cross_val_score(pipe_fs, X_train, y_train, cv=5, scoring="r2").mean()
auswahl = cross_val_score(pipe_fs, X_train[gewaehlt], y_train, cv=5, scoring="r2").mean()
print(len(gewaehlt), alle, auswahl)

### Wrapper: RFE (Recursive Feature Elimination)

Trainiert das Modell, streicht das schwächste Merkmal und wiederholt das, bis die gewünschte Anzahl übrig ist.

In [ ]:
from sklearn.feature_selection import RFE

rfe = make_pipeline(StandardScaler(), RFE(LinearRegression(), n_features_to_select=4)).fit(X_train, y_train)
print("RFE behält:", list(X.columns[rfe[-1].support_]))
print("Korrelation wählte:", gewaehlt)

**Embedded:** Lasso (siehe Ridge und Lasso oben) und die Wichtigkeiten im Random Forest wählen beim Training selbst aus.

## Hyperparameter suchen

`GridSearchCV` probiert alle Werte mit Cross-Validation auf den Trainingsdaten. Das Test-Set wird erst ganz am Schluss **einmal** benutzt.

In [ ]:
pipe_l = Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=10_000))])
grid = {"model__alpha": [0.01, 0.1, 1, 10]}
search = GridSearchCV(pipe_l, grid, cv=5, scoring="r2")
search.fit(X_train, y_train)
r2_final = r2_score(y_test, search.best_estimator_.predict(X_test))
print(search.best_params_, r2_final)

## Metriken

### Regression: MAE, MSE, R² von Hand

In [ ]:
y_true = np.array([3, 5, 2, 7])
y_pred = np.array([2.5, 5, 4, 8])

residuen = y_true - y_pred
mae = np.mean(np.abs(residuen))
mse = np.mean(residuen ** 2)
ss_res = np.sum(residuen ** 2)
ss_tot = np.sum((y_true - y_true.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
print(f"MAE={mae:.3f}  MSE={mse:.3f}  R2={r2:.3f}")
print("sklearn:", mean_absolute_error(y_true, y_pred), mean_squared_error(y_true, y_pred), round(r2_score(y_true, y_pred), 3))

### Klassifikation: aus der Confusion Matrix

Brustkrebs-Daten wie im Notebook Klassifikation. Klasse 1 = gutartig.

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=SEED)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc_train, yc_train)
pred, proba = logreg.predict(Xc_test), logreg.predict_proba(Xc_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(yc_test, pred).ravel()
print(f"TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"Precision = TP/(TP+FP) = {tp / (tp + fp):.3f}")
print(f"Recall    = TP/(TP+FN) = {tp / (tp + fn):.3f}")
print(f"Accuracy  = (TP+TN)/alle = {(tp + tn) / (tp + tn + fp + fn):.3f}")
print(f"ROC AUC (über alle Schwellen) = {roc_auc_score(yc_test, proba):.3f}")

## Data Leakage

Der klassische Fehler: Merkmale **vor** der Cross-Validation auswählen. Die Daten hier sind reiner Zufall, ehrlich ist also nur etwa 50 % möglich.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(SEED)
X_rausch = rng.normal(size=(100, 5000))
y_rausch = rng.integers(0, 2, 100)

# Wrong: selection sees all rows, including the later validation folds
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_rausch, y_rausch)
falsch = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_rausch, cv=5).mean()

# Right: selection inside the pipeline, refitted on each training fold
richtig = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)),
                          X_rausch, y_rausch, cv=5).mean()
print(f"Auswahl vor der CV (Leakage): {falsch:.2f}")
print(f"Auswahl in der Pipeline:      {richtig:.2f}")

**Deutung:** Mit Leakage sieht das Modell auf reinem Zufall sehr gut aus. In der Pipeline landet es ehrlich bei etwa 50 %. Dasselbe gilt für Skalieren, Imputation und Resampling: Alles, was aus Daten lernt, gehört in die Pipeline.